In [91]:
%pip install psycopg2


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
print ('Ola mundo')



Ola mundo


In [9]:
import psycopg2 as pg 
from datetime import datetime

In [121]:
conexao = pg.connect(dbname="centroesportivo", user="postgres", password="postgres", host="localhost", port="5432") 

In [122]:
cursor = conexao.cursor()
print("Conectado")

Conectado


In [123]:
conexao.rollback()

In [124]:
cursor.execute("SELECT * FROM cliente;")
cliente = cursor.fetchall()
print(cliente)

[]


In [232]:
conexao.rollback()

cursor.executemany(
    """
    INSERT INTO agendamento (tempo_termino_max, hora)
    VALUES (%s, %s)
    """,
    [
    (55, "20:00-2026-10-21"),
    (51, "20:00-2026-10-22"),
    (52, "20:00-2026-09-13"),
    (59, "20:00-2026-09-21"),
    (50, "20:00-2026-10-19"),
    (59, "20:00-2026-08-17"),
    (58, "20:00-2026-09-15"),
    (59, "20:00-2026-10-23"),
        
    ]
)

conexao.commit()

print("Todas as partidas foram inseridas!")

InvalidTimeZoneDisplacementValue: ERRO:  deslocamento de zona horária fora do intervalo: "20:00-2026-10-21"
LINE 3:     VALUES (55, '20:00-2026-10-21')
                        ^


In [136]:
cursor.execute("SELECT * FROM cliente;")
clientes_para_inserir = cursor.fetchall()
print(clientes_para_inserir)

[(1, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', None)]


In [194]:
conexao.rollback()
cursor.execute("SELECT * FROM cliente;")
clientes_para_inserir = cursor.fetchall()
print(clientes_para_inserir)

for cliente in clientes_para_inserir:
    print(cliente)

[(1, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', None), (2, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', '51 93425644'), (3, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', '51 93425644'), (4, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', '51 93425644')]
(1, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', None)
(2, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', '51 93425644')
(3, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', '51 93425644')
(4, 'Pedro Ricardo Souza', 'pedro.ricardo23@gmail.com', '51 93425644')


In [196]:
class Cliente():
    id_cliente:int
    nome: str
    telofone: str 
    email: str

    def __init__(self, nome:str, telefone:str, email:str):
        self.nome=nome
        self.telefone=telefone
        self.email=email
        
    def __str__(self):
        return f"Cliente(nome[{self.nome}, telefone={self.telefone}, email={self.email}])"

    def __repr__(self):
        return f"Cliente {self.nome}"


In [1]:
def criar_cliente(nome: str, telefone: str | None, email: str | None) -> Cliente | None:
        try:
            cursor.execute(
                "INSERT INTO cliente (nome, telefone, email) "
                "VALUES (%s, %s, %s) RETURNING id_cliente",
                (nome, telefone, email)
            )
            id_cliente = cursor.fetchone()[0]  # o id que o SERIAL acabou de gerar
            conexao.commit()
            return Cliente(nome, telefone, email, id_cliente)
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return None

In [2]:
def listar_clientes() -> list[Cliente]:
        clientes = []
        try:
            cursor.execute("SELECT id_cliente, nome, telefone, email FROM cliente ORDER BY id_cliente")
            for linha in cursor.fetchall():
                clientes.append(Cliente(linha[1], linha[2], linha[3], linha[0]))
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return clientes


In [3]:
def buscar_cliente(id_cliente: int) -> Cliente | None:
        try:
            cursor.execute(
                "SELECT id_cliente, nome, telefone, email FROM cliente WHERE id_cliente = %s",
                (id_cliente,)  # a vírgula faz disso uma tupla
            )
            linha = cursor.fetchone()  # None se nenhuma linha bateu com o WHERE
            if linha:
                return Cliente(linha[1], linha[2], linha[3], linha[0])
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return None

In [4]:
def atualizar_cliente(cliente: Cliente) -> bool:
        try:
            cursor.execute(
                "UPDATE cliente SET nome = %s, telefone = %s, email = %s WHERE id_cliente = %s",
                (cliente.nome, cliente.telefone, cliente.email, cliente.id_cliente)
            )
            alteradas = cursor.rowcount  # 0 = não existe cliente com esse id
            conexao.commit()
            return alteradas > 0
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return False

In [5]:
def deletar_cliente(id_cliente: int) -> bool:
        try:
            cursor.execute("DELETE FROM cliente WHERE id_cliente = %s", (id_cliente,))
            apagadas = cursor.rowcount
            conexao.commit()
            return apagadas > 0
        except pg.DatabaseError as e:
            print(e)  # ex.: o cliente tem agendamento, e a chave estrangeira impede o DELETE
            conexao.rollback()
        return False

In [139]:
class Quadra():
    id_quadra:int           
    preco: float 
    coberta: bool  
    
    def __init__(self, preco:float, coberta:bool):
        self.preco = preco
        self.coberta = coberta
        
    def __str__(self):
        return f"Quadra (preco[{self.preco}, coberta={self.coberta}])"

    def __repr__(self):
        return f"Quadra {self.preco}"


In [140]:
class Partida():
    id_partida: int
    nome_artilheiro: str
    total_gols: int

    def __init__(self, nome_artilheiro:str, total_gols:int):
        self.nome_artilheiro=nome_artilheiro
        self.total_gols=total_gols

    def __str__(self):
        return f"Partida( nome_artilheiro[{self.nome_artilheiro}, total_gols={self.total_gols}])"

    def __repr__(self):
        return f"Partida {self.nome_artilheiro}"

In [141]:

class Agendamento():
    id_agendamento: int
    id_quadra: int
    tempo_termino_max: int
    hora: datetime  
    id_partida: int


    def __init__(self, tempo_termino_max: int, hora:datetime):
        self.tempo_termino_max=tempo_termino_max
        self.hora=hora

    def __str__(self):
        return f"Agendamento( tempo_termino_max[{self.tempo_termino_max}, hora={self.hora}])"
    
    def __repr__(self):
        return f"Agendamento {self.tempo_termino_max}"

In [ ]:
clientes_para_inserir: list[Cliente] = [ 
    Cliente("Pedro Ricardo Souza", "51 93425644", "pedro.ricardo23@gmail.com"),
    Cliente("Pedro Augusto Silva", "47 98675632", "pedro.aug3412@gmail.com"),
    Cliente("Pedro Souza", "51 86655434", "pedro.souza2345@gmail.com"),
    Cliente("Ricardo José", "53 97985891", "ricardo.santos45@gmail.com"),
    Cliente("Fabricio Souza", "souzafabriciio4545@gmail.com", None),
    Cliente("Joao Santos", None, None),
    Cliente("Joao Pedro Silva", None, None),]
print(clientes_para_inserir)


[Cliente  Pedro Ricardo Souza, Cliente  Pedro Augusto Silva, Cliente  Pedro Souza, Cliente  Ricardo José, Cliente  Fabricio Souza, Cliente  Joao Santos, Cliente  Joao Pedro Silva]


In [ ]:
partidas_para_inserir: list[Partida] = [  
    Partida("William", 12),      
    Partida("Bruno", 10),
    Partida("Jose, 5"),
    Partida("Gabriel Cunha", 11),
    Partida("Leonardo", 12),]
print(partidas_para_inserir)

[Partida William, Partida Bruno, Partida None, Partida Gabriel Cunha, Partida Leonardo]


In [47]:
quadras_para_inserir: list[Quadra] = [
    Quadra("100,00", True),
    Quadra("100,00", True),
    Quadra("90,00", False), 
    Quadra("90,00", False),]
print(quadras_para_inserir)

[Quadra 100,00, Quadra 100,00, Quadra 90,00, Quadra 90,00]


In [8]:
agendamentos_para_inserir: list[Agendamento] = [
    Agendamento(55, datetime(2026, 10, 21, 20, 0)),
    Agendamento(51, datetime(2026, 10, 22, 20, 0)),
    Agendamento(52, datetime(2026, 09, 13, 20, 0)),
    Agendamento(59, datetime(2026, 09, 21, 20, 0)),
    Agendamento(50, datetime(2026, 10, 19, 20, 0)),
    Agendamento(59, datetime(2026, 08, 17, 20, 0)),
    Agendamento(58, datetime(2026, 09, 15, 20, 0)),
    Agendamento(59, dateime(2026, 10, 23, 20,0)),]
print(agendamentos_para_inserir)

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2610776477.py, line 4)